# ¿Están reverdeciendo las cumbres?

**Sierra Nevada frente a Pirineos, con Landsat y ndvi2gif · 1985-2024**

Un titular reciente decía que *"las cimas de las montañas se están volviendo verdes"*. El artículo del que salía ([Wessely et al., 2026, *Science*](https://www.science.org/doi/10.1126/science.aed2974)) es un estudio **de campo**: inventarios de la red GLORIA en 62 cumbres europeas. Lo que encuentra es que **hay más especies arriba**, porque suben las de cotas bajas, y que a la vez **se extinguen localmente las especialistas de alta montaña**.

Nosotros vamos a mirar otra cosa: **¿ha subido el NDVI de verano en el cinturón alpino?** Es una pregunta parecida pero **no es la misma**. Un satélite ve verde; no ve qué especie lo produce. Ten esa distinción en la cabeza todo el ejercicio.

**El plan:**

1. **Delimitar las cumbres de dos maneras** — por altitud, y mirando dónde aguanta la nieve en verano.
2. **Medir el NDVI máximo de verano** de cada año, de 1985 a 2024.
3. **Resumirlo por décadas** y comparar las dos cordilleras en un mismo gráfico.

## 0 · Preparativos

Si estás en Colab, ejecuta la celda de instalación. En tu propio Python, salta a la siguiente.

In [ ]:
# Solo en Google Colab
!pip install -q ndvi2gif geemap

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
from ndvi2gif import NdviSeasonality

PROYECTO = 'ee-tunombre'   # <-- pon aquí tu proyecto de Earth Engine

ee.Authenticate()
ee.Initialize(project=PROYECTO)
print('Earth Engine listo')

> Al importar ndvi2gif verás un aviso de que `COPERNICUS/DEM/GLO30` está obsoleto. Viene del módulo de radar de la librería, **no de lo que vamos a hacer aquí**: nosotros usamos el modelo nuevo. Ignóralo.

---

## 1 · ¿Dónde están las cumbres? Dos maneras de decidirlo

Antes de medir nada hay que decir **qué es cumbre**. No hay una respuesta única, y la decisión cambia el resultado. Vamos a probar dos criterios topográficos sobre el mismo modelo de elevación.

In [ ]:
# Modelo digital de elevaciones global, 30 m
DEM = ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1').select('DEM').mosaic()

# Un recuadro por cordillera: [oeste, sur, este, norte]
CAJAS = {
    'Sierra Nevada': ee.Geometry.Rectangle([-3.45, 37.00, -3.15, 37.12]),
    'Pirineos':      ee.Geometry.Rectangle([-0.10, 42.60,  0.20, 42.76]),
}

for nombre, caja in CAJAS.items():
    cima = DEM.reduceRegion(ee.Reducer.max(), caja, 30, maxPixels=1e9).get('DEM')
    print(f'{nombre}: la cima está a {round(cima.getInfo())} m')

**Debe salir:** Sierra Nevada 3.472 m (el Mulhacén) y Pirineos 3.340 m (Monte Perdido).

### Criterio A · un umbral fijo de altitud

El más sencillo: todo lo que esté por encima de una cota. Usamos **2.500 m**, que en estas latitudes anda cerca del límite del arbolado.

### Criterio B · desde la cima hacia abajo

El problema del umbral fijo es que **trata igual a cordilleras distintas**. Un criterio relativo es quedarse con los **500 m más altos de cada cordillera**: cada una se mide con su propia vara.

In [ ]:
def cinturon(caja, umbral):
    """Devuelve el polígono de todo lo que supera 'umbral' metros dentro de 'caja'."""
    return (DEM.gt(umbral).selfMask()
               .reduceToVectors(geometry=caja, scale=90,
                                geometryType='polygon', maxPixels=1e9)
               .geometry())

def km2(geom):
    return round(geom.area(maxError=30).getInfo() / 1e6, 1)

ZONAS, filas = {}, []
for nombre, caja in CAJAS.items():
    cima = round(DEM.reduceRegion(ee.Reducer.max(), caja, 30, maxPixels=1e9).get('DEM').getInfo())
    zona_a = cinturon(caja, 2500)          # criterio A
    zona_b = cinturon(caja, cima - 500)    # criterio B
    ZONAS[nombre] = {'A': zona_a, 'B': zona_b, 'cima': cima}
    filas.append({'zona': nombre, 'cima_m': cima,
                  'A: > 2500 m (km2)': km2(zona_a),
                  'B: cima - 500 m': f'> {cima - 500} m',
                  'B (km2)': km2(zona_b)})

pd.DataFrame(filas)

**Debe salir:**

| zona | cima | A: > 2.500 m | B: cima − 500 m | B |
|---|---|---|---|---|
| Sierra Nevada | 3.472 m | **161,0 km²** | > 2.972 m | **27,4 km²** |
| Pirineos | 3.340 m | **60,8 km²** | > 2.840 m | **11,6 km²** |

Fíjate en lo que acaba de pasar: con el **mismo umbral de 2.500 m**, Sierra Nevada tiene **casi el triple de superficie** que el recuadro pirenaico. No es que sea "más alta": es más **maciza** por encima de esa cota. Si comparas las dos cordilleras con el criterio A estás comparando dos cosas de tamaño muy distinto.

El criterio B las iguala mucho más (27 frente a 12 km²) pero se queda con muy poca superficie, y con pocos píxeles el ruido manda.

**No hay opción correcta.** Lo importante es que la elijas **a sabiendas** y que digas cuál has usado. Nosotros seguimos con la **A**, que tiene más píxeles y aguanta mejor el ruido.

In [ ]:
Mapa = geemap.Map(center=[39.5, -1.5], zoom=6)
Mapa.addLayer(DEM.updateMask(DEM.gt(1500)), {'min': 1500, 'max': 3500,
              'palette': ['#2c7bb6', '#ffffbf', '#d7191c']}, 'Altitud > 1500 m')
for nombre, z in ZONAS.items():
    Mapa.addLayer(z['A'], {'color': 'blue'},  f'{nombre} · A (> 2500 m)')
    Mapa.addLayer(z['B'], {'color': 'red'},   f'{nombre} · B (cima - 500)')
Mapa

---

## 2 · La otra manera de delimitar: dónde aguanta la nieve

La altitud es una aproximación. Lo que de verdad define el ambiente alpino es **cuánto dura la nieve**, y eso se puede medir: el **NDSI** (índice de nieve) separa la nieve del suelo y la roca.

Aquí es donde entra **ndvi2gif**: le pedimos el **percentil 95 del NDSI de verano**, o sea *cómo de nevado llega a estar el píxel en su mejor momento del verano*. Si un píxel tiene NDSI alto **en verano**, es que la nieve le dura.

In [ ]:
def ndsi_verano(zona, anio=2020):
    """Percentil 95 del NDSI de verano. Ojo: con key='percentile' la banda se llama 'nd_p95'."""
    obj = NdviSeasonality(roi=zona, sat='Landsat', index='ndsi', periods=4,
                          key='percentile', percentile=95,
                          start_year=anio, end_year=anio, max_cloud_cover=60)
    return obj.get_period_composite(anio, 2)     # 2 = verano (julio-septiembre)

for nombre, z in ZONAS.items():
    img = ndsi_verano(z['A'])
    mediana = img.reduceRegion(ee.Reducer.median(), z['A'], 60,
                               maxPixels=1e9, bestEffort=True).get('nd_p95').getInfo()
    nieve = img.gt(0.4).multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(), z['A'], 60, maxPixels=1e9, bestEffort=True).get('nd_p95').getInfo() / 1e6
    print(f'{nombre}: NDSI p95 mediano {mediana:6.3f} | superficie con NDSI > 0,4: {nieve:5.1f} km2')

**Debe salir** (verano de 2020):

| zona | NDSI p95 mediano | superficie con NDSI > 0,4 |
|---|---|---|
| Sierra Nevada | −0,137 | **0,5 km²** |
| Pirineos | −0,215 | **5,2 km²** |

Y aquí tienes el primer resultado de verdad del ejercicio: **en los Pirineos queda diez veces más superficie con nieve de verano que en Sierra Nevada**, aunque el cinturón pirenaico es casi tres veces más pequeño.

Dos cordilleras por encima de la misma cota, con ambientes muy distintos. Eso ya anticipa que **no tienen por qué reverdecer igual**.

> **Para pensar:** la mediana del NDSI p95 es negativa en las dos. ¿Qué significa eso? ¿Y por qué el umbral de 0,4 es una decisión tan discutible como la de los 2.500 m?

---

## 3 · El NDVI máximo de verano, año a año

Ahora la medida principal. Para cada año:

1. ndvi2gif construye el **composite de verano**: para cada píxel, el **NDVI máximo** de julio a septiembre.
2. De ese mapa sacamos **un número por año**: el **percentil 95 espacial**, es decir, cómo de verde llega a estar la parte más verde del cinturón.

Usamos el percentil 95 y no el máximo porque un solo píxel raro no debe decidir el año entero.

In [ ]:
ANIOS = list(range(1985, 2025))
VERANO = 2

def serie_verano(zona):
    """Un valor por año, calculado entero en el servidor: un solo getInfo."""
    obj = NdviSeasonality(roi=zona, sat='Landsat', index='ndvi', periods=4,
                          key='max', start_year=1985, end_year=2024,
                          max_cloud_cover=60)
    imgs = ee.ImageCollection([obj.get_period_composite(a, VERANO).set('anio', a)
                               for a in ANIOS])

    def resumen(img):
        v = img.reduceRegion(ee.Reducer.percentile([95]), zona, 60,
                             maxPixels=1e9, bestEffort=True)
        # Con key='max' la banda se llama 'nd'. El -999 cubre un verano sin escenas.
        return ee.Feature(None, {'anio': img.get('anio'), 'ndvi': v.get('nd', -999)})

    fc = ee.FeatureCollection(imgs.map(resumen)).getInfo()
    df = pd.DataFrame([f['properties'] for f in fc['features']]).sort_values('anio')
    df['ndvi'] = df['ndvi'].where(df['ndvi'] > -1)    # -999 -> vacío
    return df.set_index('anio')['ndvi']

series = {nombre: serie_verano(z['A']) for nombre, z in ZONAS.items()}
serie = pd.DataFrame(series)
print('Años sin datos:', serie.index[serie.isna().any(axis=1)].tolist())
serie.head()

Tarda unos segundos, no minutos, y eso no es casualidad: **todo el cálculo ocurre en los servidores de Google** y solo baja el resultado final. Si hicieras un `getInfo()` por año dentro de un bucle de Python, cada vuelta sería un viaje de ida y vuelta a Google y esto tardaría **más de diez minutos**.

---

## 4 · De cuarenta años a cuatro décadas

Cuarenta valores anuales tienen mucho ruido: un verano con nieve tardía o con pocas escenas útiles sube o baja la línea sin que haya pasado nada en el suelo. Agrupar por décadas suaviza eso.

Para cada década nos quedamos con el **máximo de los percentiles 95 anuales**: el verano más verde de esos diez años.

In [ ]:
DECADAS = [(1985, 1994), (1995, 2004), (2005, 2014), (2015, 2024)]

tabla = pd.DataFrame(
    {nombre: {f'{ini}-{fin}': round(float(s.loc[ini:fin].max()), 3)
              for ini, fin in DECADAS}
     for nombre, s in series.items()}
)
tabla

**Debe salir:**

| década | Pirineos | Sierra Nevada |
|---|---|---|
| 1985-1994 | 0,425 | 0,395 |
| 1995-2004 | 0,572 | 0,428 |
| 2005-2014 | 0,527 | 0,402 |
| 2015-2024 | **0,613** | **0,479** |

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colores = {'Sierra Nevada': '#d7191c', 'Pirineos': '#2c7bb6'}
etiquetas = [f'{i}-{f}' for i, f in DECADAS]

for nombre in tabla.columns:
    ax.plot(etiquetas, tabla[nombre], 'o-', lw=2.5, ms=9,
            color=colores[nombre], label=nombre)
    for x, y in zip(etiquetas, tabla[nombre]):
        ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9, color=colores[nombre])

ax.set_ylabel('NDVI de verano\n(máximo de los percentiles 95 anuales)')
ax.set_title('¿Reverdecen las cumbres? Cinturón por encima de 2.500 m')
ax.grid(alpha=.3)
ax.legend()
fig.tight_layout()

In [ ]:
# Y la serie anual completa, por si quieres ver el ruido que hemos suavizado
fig, ax = plt.subplots(figsize=(11, 4.5))
for nombre in serie.columns:
    ax.plot(serie.index, serie[nombre], 'o-', ms=3, lw=1,
            color=colores[nombre], alpha=.8, label=nombre)
ax.set_ylabel('NDVI de verano (p95)')
ax.set_xlabel('año')
ax.grid(alpha=.3)
ax.legend()
fig.tight_layout()

---

## 5 · Qué hemos visto, y qué no

**Las dos cordilleras suben**, y el Pirineo sube más: de 0,425 a 0,613 (**+0,188**) frente a 0,395 a 0,479 en Sierra Nevada (**+0,084**), algo más del doble.

Pero fíjate en la tercera década: **las dos bajan** entre 1995-2004 y 2005-2014, y luego vuelven a subir. No es una recta.

**Preguntas para cerrar. Responde en tres frases:**

1. ¿Dirías que las cumbres reverdecen? ¿Con qué seguridad, y para cuál de las dos cordilleras?
2. Hemos delimitado las cumbres de tres maneras distintas (dos altitudinales y una por nieve) y hemos escogido una. ¿Cuánto crees que cambiaría el resultado con otra? **Compruébalo**: repite el cálculo con `z['B']` y compara.
3. El artículo de *Science* dice que en las cumbres hay **más especies y a la vez más extinciones locales** de las especialistas alpinas. Si eso está pasando aquí, **¿cómo se vería en este gráfico?** ¿Y podríamos distinguirlo de que simplemente crezcan más las mismas plantas de siempre?

La tercera es la importante. El NDVI sube en los dos casos.